#### Analyze crawler output for RA

In [ ]:
from pathlib import Path
import pandas as pd


import monai
from monai.data import Dataset, CacheDataset, DataLoader

import monai
from monai.data import PILReader
from monai.transforms import LoadImage, LoadImaged, Resized, Compose, SaveImage, Spacingd, SpatialCropd, ResizeWithPadOrCropd


import numpy as np

In [ ]:

def extract_extras_from_filename(filename: str): 
    x = filename.split(".dcm")[0].split("_")
    d = {
        "filename": filename, 
        "id": x[0],
        "date_str": x[1],
        "sex": x[2],
        "left_or_right": x[3],
        # I dont know what the rest means:  dp_MTwo_InvNo_RotNo_BOk_OPNo_app_ComNo
        "x4": x[4],
        "x5": x[5],
        "x6": x[6],
        "x7": x[7],
        "x8": x[8],     
        "x9": x[9],    
        "x10": x[10],    
        "x11": x[11]                                        
    }
    return d

def extract_extras_from_abspath(abs_path):
    filename = str(abs_path).split(str("/"))[-1]
    return {**{"image": abs_path, **extract_extras_from_filename(filename)}}



In [ ]:


# file = "/home/cwatzenboeck/data/AutoPIX_local_data/tabular/metadata_crawler/Arthritis_crawler.csv"

# file = "/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora/autoscoRA_data/autoscoRA_feet.csv"
# df = pd.read_csv(file)

#base_dir = Path("/project/autoscora/")
base_dir = Path("/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora")

folder_H = base_dir / "autoscoRA_images/H_images_of_interest_2_renamed_mirrored_inverted_dicoms"
folder_F = base_dir / "autoscoRA_images/F_images_of_interest_2_renamed_mirrored_inverted_dicoms"


folder_H = Path(folder_H)
folder_F = Path(folder_F)


In [ ]:
files_F = list(folder_F.glob("*.dcm"))
#files_H = list(folder_H.glob("*.dcm"))

In [ ]:
len(files_F)#, len(files_H)

In [ ]:
# I want to load a dcm image (CR 2d) with monai. Improve the code below 


files_F_with_extras = [{"image": str(extract_extras_from_abspath(file)["image"])} for file in files_F]

transform = Compose([
    LoadImaged(keys=["image"], ensure_channel_first=True, reader="PydicomReader"),
    #Resized(keys=["image"], spatial_size=(512,512))
    #Spacingd(keys=["image"], pixdim=(, 1.0e-2), mode="bilinear"),
    #ResizeWithPadOrCropd(keys=["image"], spatial_size=(128, 128))  # Adjust spatial_size as required
])


dataset = Dataset(files_F_with_extras, transform=transform)
dataloader = DataLoader(dataset, batch_size=1, num_workers=2)

X = next(iter(dataloader))


# TODO plot X["image"]

In [ ]:
import matplotlib.pyplot as plt
import torch

# Assuming X["image"] is a torch.Tensor of shape (B, C, H, W)
images = X["image"]

# Convert tensor to numpy array if needed
if isinstance(images, torch.Tensor):
    images = images.numpy()

# Loop over each image in the batch and plot
for i in range(images.shape[0]):
    img = images[i]  # shape: (C, H, W)
    # If the image has only one channel, remove the channel dimension for plotting
    if img.shape[0] == 1:
        img = img.squeeze(0)
    plt.figure()
    plt.imshow(img, cmap="gray")
    plt.title(f"Image {i}")
    plt.axis("off")

plt.show()
